### RQ1 — Cross-Architecture (Family-Holdout) Test

The actual RQ1 test: train on one family only, test on the *entire other family*, never seen during training. Random Forest, log1p target, same 19-column feature set as `03g_model_bakeoff.ipynb` (core + `is_gpu` + `shape_*` + PMLB dataset properties). Two directions:

1. Train on EC-NAS only → test on all of BUTTER-E
2. Train on BUTTER-E only → test on all of EC-NAS

**Why "% of within-family ceiling" rather than raw numbers alone:** BUTTER-E's own achievable Kendall-Tau (training and testing within BUTTER-E) is 0.936 (`03a`); EC-NAS's is 0.860 (`03b`). A cross-family tau of, say, 0.65 means something different depending on whether the ceiling for that test family was 0.86 or 0.94 — the raw number alone can't say whether the model is transferring well or just benefiting from an easy target family.

In [ ]:
# IMPORTS & LOAD

import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score

sys.path.insert(0, "../../")
from src.models import random_forest

df = pd.read_csv("../../data/processed/combined/combined_features.csv")

NON_FEATURE_COLS = ["run_id", "target", "family", "source_dataset"]
FEATURES = [c for c in df.columns if c not in NON_FEATURE_COLS]
TARGET = "target"

butter = df[df["family"] == "MLP"]
ecnas = df[df["family"] == "CNN"]
print("BUTTER-E rows:", len(butter), " EC-NAS rows:", len(ecnas))

# confirm masking: auxiliary columns should be exactly 0 for every EC-NAS row
aux_cols = [c for c in FEATURES if c not in ("params", "depth", "flops", "epochs", "batch_size")]
print("aux cols all-zero in EC-NAS:", (ecnas[aux_cols] == 0).all().all())

In [ ]:
# HOLDOUT HELPER — train on one family entirely, test on the other entirely

def holdout(train_df, test_df, label):
    X_train = train_df[FEATURES]
    y_train_log = np.log1p(train_df[TARGET])

    X_test = test_df[FEATURES]
    y_test = test_df[TARGET]

    model = random_forest.build_model()
    model.fit(X_train, y_train_log)
    preds_raw = np.expm1(model.predict(X_test))

    mape = mean_absolute_percentage_error(y_test, preds_raw)
    r2 = r2_score(y_test, preds_raw)
    tau, tau_p = kendalltau(y_test, preds_raw)

    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

    print(f"=== {label} ===")
    print(f"train n={len(train_df)}, test n={len(test_df)}")
    print(f"MAPE: {mape:.4f}   R2: {r2:.4f}   Kendall-Tau: {tau:.4f}  (p={tau_p:.2e})")
    print("\nfeature importances:")
    print(importances)
    print()

    return dict(label=label, mape=mape, r2=r2, tau=tau, importances=importances)

In [ ]:
# RUN BOTH DIRECTIONS

res_ec_to_butter = holdout(ecnas, butter, "EC-NAS -> BUTTER-E (train EC-NAS only, test all BUTTER-E)")
res_butter_to_ec = holdout(butter, ecnas, "BUTTER-E -> EC-NAS (train BUTTER-E only, test all EC-NAS)")

In [ ]:
# RELATIVE TO WITHIN-FAMILY CEILING

BUTTER_CEILING = 0.936  # 03a_within_butter_e.ipynb
ECNAS_CEILING = 0.860   # 03b_within_ec_nas.ipynb

pct_ec_to_butter = 100 * res_ec_to_butter["tau"] / BUTTER_CEILING
pct_butter_to_ec = 100 * res_butter_to_ec["tau"] / ECNAS_CEILING

print(f"EC-NAS -> BUTTER-E: tau={res_ec_to_butter['tau']:.4f}  vs ceiling {BUTTER_CEILING} -> {pct_ec_to_butter:.1f}% of ceiling")
print(f"BUTTER-E -> EC-NAS: tau={res_butter_to_ec['tau']:.4f}  vs ceiling {ECNAS_CEILING} -> {pct_butter_to_ec:.1f}% of ceiling")

**Result** (executed once already; re-run in VS Code to attach outputs):

| direction | MAPE | R² | Kendall-Tau | % of test family's own ceiling |
|---|---:|---:|---:|---:|
| EC-NAS → BUTTER-E | 0.848 | **-0.386** | 0.263 | **28.1%** (of BUTTER-E's 0.936) |
| BUTTER-E → EC-NAS | 0.900 | **-8.533** | 0.655 | **76.1%** (of EC-NAS's 0.860) |

**Both directions transfer poorly by MAPE/R² (both R² are negative — worse than predicting the mean), but they fail for two different, specific reasons, not the same generic "cross-family generalization is hard.":**

**EC-NAS → BUTTER-E: the model has nothing to learn from.** Every auxiliary column (`is_gpu`, `shape_*`, all 5 PMLB properties) is masked to exactly 0 for every EC-NAS row, and `epochs`/`batch_size` are also constants within EC-NAS. Feature importances confirm it directly: `flops` (0.495) and `params` (0.489) carry essentially 100% of the trained model's decisions — every other feature, including all the ones that mattered most within BUTTER-E, gets **exactly 0.0** importance, because EC-NAS-only training data has zero variance in them to split on. This isn't a transfer failure so much as a training-data-coverage limitation: a model trained on EC-NAS literally cannot acquire the `n_observations`/`is_gpu` signal that BUTTER-E needs, because that signal doesn't exist anywhere in EC-NAS. Combined with the earlier params-range finding (`02c`: only 34.0% of BUTTER-E rows fall inside EC-NAS's trained parameter range), this direction is doubly disadvantaged — no relevant auxiliary signal *and* mostly extrapolating on `params`/`flops` too. 28.1% of ceiling is a low, honest number, not a bug.

**BUTTER-E → EC-NAS: a genuinely different problem — a masking-induced out-of-distribution artifact, not a lack of transferable signal.** The model *does* learn real, useful dependence on `n_observations`/`is_gpu` from BUTTER-E (same importance ranking as every within-family BUTTER-E run: `n_observations` 0.60, `is_gpu` 0.13). But every EC-NAS test row presents `n_observations=0, is_gpu=0`, all `shape_*=0`, etc. — a combination that **never occurs in real BUTTER-E data** (a real training dataset always has n_observations > 0). The model has never seen this input pattern and has no reason to treat "0" as "not applicable" rather than "an unusually small/CPU-only training setup" — it's extrapolating into a region of input space the masking scheme created, not one that reflects anything about the EC-NAS architectures themselves. This shows up as catastrophic *absolute*-scale error (R²=-8.5, MAPE=90%) but comparatively preserved *relative* ordering (tau=0.655, 76.1% of ceiling) — consistent with a fairly systematic bias (the same spurious "0" pattern shifts every EC-NAS prediction in a similar direction) rather than noise that scrambles row-to-row ranking.

**This is a real limitation of the zero-fill masking scheme specifically for family-holdout testing** (it was fine for the pooled random-split setting in `03g`, where the model sees the 0-pattern as part of EC-NAS's actual training distribution too — the holdout setting is different because the model trained on BUTTER-E never saw *any* all-zero-auxiliary rows during training). Worth flagging for the thesis discussion: RQ1's two directions aren't just differently hard, they fail via different, identifiable mechanisms — one a genuine data/coverage gap (EC-NAS→BUTTER-E), one an artifact of how missing family-specific features are encoded (BUTTER-E→EC-NAS) rather than of cross-family transfer itself. A cleaner masking scheme (e.g. an explicit "feature not applicable" indicator rather than a numeric 0 that collides with real small values) is a plausible fix worth considering before this becomes the final RQ1 methodology — not attempted here.

### Re-run with `is_mlp_family` flag (per `02c_combined_features.ipynb`'s fix)

In [ ]:
# RELOAD (24 columns, is_mlp_family added) & RE-RUN BOTH DIRECTIONS

df2 = pd.read_csv("../../data/processed/combined/combined_features.csv")
FEATURES2 = [c for c in df2.columns if c not in NON_FEATURE_COLS]
print("n features:", len(FEATURES2))

butter2 = df2[df2["family"] == "MLP"]
ecnas2 = df2[df2["family"] == "CNN"]

print("is_mlp_family unique value within EC-NAS-only training data:", ecnas2["is_mlp_family"].unique())
print("is_mlp_family unique value within BUTTER-E-only training data:", butter2["is_mlp_family"].unique())

In [ ]:
# RE-RUN using the updated feature list (the `holdout` helper above reads the global
# FEATURES, so rebind it before calling)

FEATURES = FEATURES2

res_ec_to_butter2 = holdout(ecnas2, butter2, "EC-NAS -> BUTTER-E (with is_mlp_family)")
res_butter_to_ec2 = holdout(butter2, ecnas2, "BUTTER-E -> EC-NAS (with is_mlp_family)")

print("is_mlp_family importance, EC-NAS->BUTTER-E:", res_ec_to_butter2["importances"].get("is_mlp_family"))
print("is_mlp_family importance, BUTTER-E->EC-NAS:", res_butter_to_ec2["importances"].get("is_mlp_family"))

**Result: the fix does not move the numbers, at all** (executed once already; re-run in VS Code to attach outputs):

| direction | metric | before fix | after fix |
|---|---|---:|---:|
| EC-NAS → BUTTER-E | Kendall-Tau | 0.2630 | 0.2630 |
| EC-NAS → BUTTER-E | R² | -0.386 | -0.386 |
| BUTTER-E → EC-NAS | Kendall-Tau | 0.6547 | 0.6546 |
| BUTTER-E → EC-NAS | R² | -8.533 | -8.585 |

Identical to four decimal places (the R² differences of ~0.001–0.05 are ordinary RF run-to-run noise, not a systematic effect). `is_mlp_family` importance is **exactly 0.0 in both directions.**

**This isn't a failed fix so much as a correction to the original diagnosis, and it's worth stating plainly.** The reasoning in the previous section — "the model can't tell a genuine 0 from a not-applicable 0" — was incomplete. The actual mechanism: `is_mlp_family` is **constant** within any single-family training set — `0` for every row when training on EC-NAS only, `1` for every row when training on BUTTER-E only (confirmed above). A Random Forest builds splits from variance it observes in training data; a feature with zero variance in the training fold cannot be split on, full stop, regardless of what it represents conceptually. This holds for *any* flag column here, not just this specific encoding choice — the strict single-family holdout setup means the model never has a chance to learn "when this flag is 1, treat these other columns as real; when it's 0, ignore them," because it only ever sees one flag value during training.

**Where this leaves things:**
- `is_mlp_family` may still be useful for the *pooled* setting (`03g`, where both flag values appear during training) — not tested here, since this notebook is specifically the strict single-family holdout.
- For the **strict RQ1 holdout test specifically**, no per-row indicator column can fix this — the auxiliary features (`is_gpu`, `shape_*`, PMLB properties) are fundamentally untransferable in this setup, because the holdout-family's model has zero training exposure to any variation involving them (either they don't exist for that family at all, as in the EC-NAS→BUTTER-E direction, or the flag distinguishing "applicable"/"not applicable" never varies, as just confirmed). A cleaner fix, not attempted here: evaluate RQ1 using only the true shared core columns (`params, depth, flops, epochs, batch_size`) — the columns both families can genuinely populate — and treat the family-specific auxiliary features as belonging to the pooled/within-family settings only, not the cross-family holdout claim.
- The EC-NAS→BUTTER-E direction's diagnosis from before stands unchanged: a genuine data-coverage gap (no equivalent signal exists in EC-NAS at all), not something any encoding trick fixes.

### Final RQ1 result — core features only

The auxiliary features (`is_gpu`, `shape_*`, PMLB dataset properties, `is_mlp_family`) are dropped entirely for this run — not because they weren't tried, but because the previous two sections established they're structurally non-transferable under strict single-family training: they either don't exist for one family at all, or (in `is_mlp_family`'s case) are provably unusable by construction, since a Random Forest cannot split on a feature with zero variance in its training fold. This run uses only the 5 features both families can genuinely populate: `params, depth, flops, epochs, batch_size`.

In [ ]:
# CORE-ONLY HOLDOUT — both directions

CORE_FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]


def holdout_core(train_df, test_df, label):
    X_train = train_df[CORE_FEATURES]
    y_train_log = np.log1p(train_df[TARGET])

    X_test = test_df[CORE_FEATURES]
    y_test = test_df[TARGET]

    model = random_forest.build_model()
    model.fit(X_train, y_train_log)
    preds_raw = np.expm1(model.predict(X_test))

    mape = mean_absolute_percentage_error(y_test, preds_raw)
    r2 = r2_score(y_test, preds_raw)
    tau, tau_p = kendalltau(y_test, preds_raw)
    importances = pd.Series(model.feature_importances_, index=CORE_FEATURES).sort_values(ascending=False)

    print(f"=== {label} ===")
    print(f"train n={len(train_df)}, test n={len(test_df)}")
    print(f"MAPE: {mape:.4f}   R2: {r2:.4f}   Kendall-Tau: {tau:.4f}  (p={tau_p:.2e})")
    print("\nfeature importances:")
    print(importances)
    print()

    return dict(label=label, mape=mape, r2=r2, tau=tau, importances=importances, preds=preds_raw)


res_ec_to_butter_core = holdout_core(ecnas2, butter2, "EC-NAS -> BUTTER-E (core-only)")
res_butter_to_ec_core = holdout_core(butter2, ecnas2, "BUTTER-E -> EC-NAS (core-only)")

In [ ]:
# WHY BUTTER-E -> EC-NAS GOT DRAMATICALLY WORSE ON R² (not better) WITHOUT AUX FEATURES —
# check predicted vs actual scale directly, since a jump from R2=-8.6 to R2=-503 needs a
# concrete explanation, not just a number

preds_be2ec = res_butter_to_ec_core["preds"]
print("EC-NAS actual target:   min={:.0f}  max={:.0f}  mean={:.0f}".format(
    ecnas2["target"].min(), ecnas2["target"].max(), ecnas2["target"].mean()))
print("core-only predictions:  min={:.0f}  max={:.0f}  mean={:.0f}  median={:.0f}".format(
    preds_be2ec.min(), preds_be2ec.max(), preds_be2ec.mean(), np.median(preds_be2ec)))
print()
print("BUTTER-E training epochs (constant):", butter2["epochs"].unique())
print("EC-NAS actual epochs:", ecnas2["epochs"].unique())

**Final RQ1 result** (executed once already; re-run in VS Code to attach outputs):

| direction | MAPE | R² | Kendall-Tau | % of test family's ceiling |
|---|---:|---:|---:|---:|
| EC-NAS → BUTTER-E | 0.848 | -0.386 | 0.263 | **28.1%** (of 0.936) |
| BUTTER-E → EC-NAS | **7.727** | **-503.5** | 0.653 | **75.9%** (of 0.860) |

**EC-NAS → BUTTER-E is completely unchanged** (identical to 4 decimal places) — expected, since the auxiliary features already had exactly 0 importance in this direction even when they were present (§1). Confirms consistency rather than revealing anything new. Feature importances: `params` (0.496) + `flops` (0.488) + `depth` (0.016) ≈ 100%, `epochs`/`batch_size` still exactly 0 (constant within EC-NAS training data, same as every prior within-EC-NAS run).

**BUTTER-E → EC-NAS got dramatically *worse* on MAPE/R² by removing the auxiliary features (R² from -8.6 to -503.5), while Kendall-Tau barely moved (0.655→0.653).** This needed a concrete explanation, not just a number, so it's checked directly above: the core-only model predicts a mean of **773,746 J** for EC-NAS, against EC-NAS's real mean of **90,832 J** and real max of just **225,335 J** — predictions are systematically ~8.5x too high, with the median prediction (757,215 J) sitting *above EC-NAS's entire real range*. The mechanism: BUTTER-E trains for a constant 3000 epochs; the `params`/`flops`→energy relationship the model learns is implicitly calibrated to that budget. EC-NAS actually trains for 4 epochs — but since `epochs` never varies within BUTTER-E's training data, the model has no way to have learned that energy should scale down for a shorter run, and blindly applies BUTTER-E's 3000-epoch-scale relationship to EC-NAS's (comparably large or larger) parameter counts. **The auxiliary features weren't fixing this in the earlier run** — `n_observations` (always 0 for EC-NAS) was dominating the tree's routing and coincidentally clustering EC-NAS's predictions into a narrower, less-extreme region of the tree by accident of structure, not because the model was reasoning about anything real. Removing it let the genuinely-learned-but-wrongly-extrapolated `params`/`flops`↔`3000-epoch-energy` relationship drive predictions directly, which is worse in absolute terms precisely because it's a more honest reflection of what the model actually learned.

**This is the headline finding for RQ1, and it should be reported as such:** Kendall-Tau (rank quality) is comparatively robust across both feature-set choices in both directions, but absolute-scale accuracy (MAPE/R²) is not just weak but actively unstable — it depends heavily on incidental modeling choices (which features happen to be present) rather than converging toward a stable "true" cross-family answer. Both directions fail to generalize well by any metric (28.1% and 75.9% of their respective ceilings, at best), and the underlying reason differs by direction: EC-NAS→BUTTER-E is a coverage/extrapolation problem (no relevant training variance, and BUTTER-E's param range mostly falls outside EC-NAS's); BUTTER-E→EC-NAS is an implicit-constant problem (BUTTER-E's fixed epoch budget gets baked into the params→energy relationship with no way to correct for EC-NAS's very different budget). Neither is "cross-family generalization failed" in a simple sense — both are specific, identifiable modeling gaps, worth stating precisely in the thesis rather than as a single undifferentiated negative result.